# Phase 2 — Semantic Vector Search with Google Gemini Embeddings

This notebook:
1. Saves the `tag_description` column to a plain-text file.
2. Embeds each book description using Google's `text-embedding-004` model.
3. Loads all vectors into a **ChromaDB** vector store.
4. Demonstrates similarity search with a natural-language query.

**Prerequisites:** Run `1-data-exploration.ipynb` first to produce `data/books_cleaned.csv`.

**Output:** `data/tagged_description.txt` + a local Chroma DB directory `data/chroma_db/`

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()  # loads GOOGLE_API_KEY from .env

## 1. Load cleaned data

In [ ]:
df = pd.read_csv('../data/books_cleaned.csv')
print(f'Loaded {len(df)} books')
df[['isbn13', 'title', 'tag_description']].head(3)

## 2. Save `tag_description` to a text file

One book per line — no header, no index. Each line is:
```
<isbn13> <full description text>
```

In [ ]:
tagged_path = '../data/tagged_description.txt'
df['tag_description'].to_csv(tagged_path, index=False, header=False)
print(f'Saved {len(df)} tagged descriptions to {tagged_path}')

## 3. Load & split the text file into one Document per book

In [ ]:
loader = TextLoader(tagged_path, encoding='utf-8')
raw_documents = loader.load()

# chunk_size=0 + separator='\n' → one chunk per newline (= one book)
splitter = CharacterTextSplitter(chunk_size=0, chunk_overlap=0, separator='\n')
documents = splitter.split_documents(raw_documents)

print(f'Total document chunks: {len(documents)}')
print('First chunk preview:')
print(documents[0].page_content[:120], '...')

## 4. Embed with Google Gemini `text-embedding-004`

`task_type="retrieval_document"` tells the model these are *documents* being indexed (as opposed to a *query*). Gemini uses this hint to optimise embedding quality for retrieval.

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004',
    task_type='retrieval_document',
    google_api_key=os.getenv('GOOGLE_API_KEY')
)

# Build the Chroma vector store and persist it to disk
# This step embeds all ~6000+ books — takes a few minutes
db = Chroma.from_documents(
    documents,
    embeddings,
    persist_directory='../data/chroma_db'
)
print(f'Vector store built and persisted. Collection size: {db._collection.count()}')

## 5. Test similarity search

At query time we use `task_type="retrieval_query"` so Gemini optimises the query embedding for matching against the indexed documents.

In [ ]:
# Query-time embeddings use a different task_type
query_embeddings = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004',
    task_type='retrieval_query',
    google_api_key=os.getenv('GOOGLE_API_KEY')
)

# Load the persisted DB with query-optimised embeddings
db_query = Chroma(
    persist_directory='../data/chroma_db',
    embedding_function=query_embeddings
)

query = 'a story about forgiveness and redemption'
results = db_query.similarity_search(query, k=5)

for i, doc in enumerate(results):
    isbn = doc.page_content.split(' ')[0]
    preview = ' '.join(doc.page_content.split(' ')[1:60])
    print(f'[{i+1}] ISBN: {isbn}')
    print(f'    {preview}...')
    print()

In [ ]:
# Cross-reference the ISBNs back to book titles
result_isbns = [int(doc.page_content.split(' ')[0]) for doc in results]
df[df['isbn13'].isin(result_isbns)][['title', 'authors', 'categories']]